# Predictive Quality Control: EWMA vs Machine Learning

This notebook compares **EWMA Control Charts** with **XGBoost ML** for early fault detection on the NASA C-MAPSS turbofan engine datasets.

| Dataset | Operating Conditions | Fault Modes | Key Insight |
|---------|---------------------|-------------|-------------|
| **FD001** | 1 | 1 (HPC only) | EWMA catches all failures |
| **FD003** | 1 | 2 (HPC + HPT) | EWMA catches Fault Mode 1, **misses** Fault Mode 2 |

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import data_processing, spc, modeling, explainability, utils

utils.ensure_output_dirs()
sns.set_theme(style="whitegrid", palette="muted")
print("Setup complete.")

## FD001: Single Fault Mode

In [ ]:
train_fd001, test_fd001 = data_processing.prepare_dataset("FD001")
print(f"Train shape: {train_fd001.shape}")
print(f"Test  shape: {test_fd001.shape}")
train_fd001.head()

In [ ]:
# Sensor 12 trend for training engine 1
eng1 = train_fd001[train_fd001["engine_id"] == 1]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eng1["cycle"], eng1["sensor_12"], linewidth=1)
ax.set_title("FD001 — Engine 1 — sensor_12")
ax.set_xlabel("Cycle")
ax.set_ylabel("sensor_12")
plt.tight_layout()
plt.show()

In [ ]:
# EWMA on training engine 1
result_fd001 = spc.run_ewma_analysis(eng1, "sensor_12")
print(f"Breach cycle: {result_fd001['breach_cycle']}")
spc.plot_ewma_matplotlib(result_fd001, engine_id=1,
                         save_path="outputs/plots/notebook_fd001_ewma_eng1.png")
from IPython.display import Image
Image(filename="outputs/plots/notebook_fd001_ewma_eng1.png")

## FD003: Two Fault Modes

In [ ]:
train_fd003, test_fd003 = data_processing.prepare_dataset("FD003")
print(f"Train shape: {train_fd003.shape}")
print(f"Test  shape: {test_fd003.shape}")

In [ ]:
# Sensor 12 for two different engines (different fault modes)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, eng_id in enumerate([1, 11]):
    eng = train_fd003[train_fd003["engine_id"] == eng_id]
    axes[idx].plot(eng["cycle"], eng["sensor_12"], linewidth=1)
    axes[idx].set_title(f"FD003 — Engine {eng_id} — sensor_12")
    axes[idx].set_xlabel("Cycle")
    axes[idx].set_ylabel("sensor_12")

plt.tight_layout()
plt.show()

In [ ]:
# EWMA on both engines — compare breach results
for eng_id in [1, 11]:
    eng = train_fd003[train_fd003["engine_id"] == eng_id]
    result = spc.run_ewma_analysis(eng, "sensor_12")
    print(f"Engine {eng_id} — Breach: {result['breach_cycle']}")
    spc.plot_ewma_matplotlib(
        result, engine_id=eng_id,
        save_path=f"outputs/plots/notebook_fd003_ewma_eng{eng_id}.png"
    )

## ML Models — FD001 vs FD003

In [ ]:
# Train and evaluate both models
for name, train_df, test_df in [("FD001", train_fd001, test_fd001),
                                 ("FD003", train_fd003, test_fd003)]:
    X_train, y_train = modeling.prepare_features_targets(train_df)
    X_test,  y_test  = modeling.prepare_features_targets(test_df)
    model = modeling.train_model(X_train, y_train, name)
    metrics = modeling.evaluate_model(
        model, X_test, y_test, name,
        save_path=f"outputs/reports/{name}_classification_report.txt"
    )
    print(f"{name} Metrics: {metrics}\n")

## Business Value Comparison

In [ ]:
# Compare business value for one engine from each dataset
for name, test_df in [("FD001", test_fd001), ("FD003", test_fd003)]:
    eng_id = test_df["engine_id"].unique()[0]
    df_eng = test_df[test_df["engine_id"] == eng_id].copy()
    actual_fail = df_eng["cycle"].max() + df_eng["RUL"].iloc[-1]

    # EWMA
    ewma_res = spc.run_ewma_analysis(df_eng, "sensor_12")
    # ML
    model = modeling.load_model(name)
    ml_warn, _ = modeling.predict_failure_start(model, df_eng)

    bv = utils.calculate_business_value(
        eng_id, name, actual_fail,
        ewma_res["breach_cycle"], ml_warn
    )
    print(utils.format_business_value_report(bv))
    print()